In [2]:
from typing import TypedDict, Optional
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langgraph.store.base import BaseStore
from langgraph.store.memory import InMemoryStore

# 1. Short-term state schema (Thread localized)
class ChatState(TypedDict):
    user_id: str
    user_input: str
    agent_response: str

# 2. Worker node that blends Short-term conversation with Long-term profiles
def smart_assistant_node(state: ChatState, store: BaseStore):
    user_id = state["user_id"]
    user_input = state["user_input"]
    
    # --- FETCH LONG-TERM MEMORY ---
    # Look up the persistent profile for this user from the 'preferences' namespace
    namespace = ("preferences", user_id)
    profile_item = store.get(namespace, key="user_profile")
    
    # Blend long-term knowledge into the execution prompt if it exists
    system_context = "You are a helpful assistant."
    if profile_item and profile_item.value:
        saved_prefs = profile_item.value
        system_context += f" User Profile Context: Prefer format: {saved_prefs.get('format', 'any')}, Tone: {saved_prefs.get('tone', 'casual')}."
    
    print(f" [System Log] Active System Context: {system_context}")
    
    # --- DETECT CRITICAL USER INSIGHTS TO SAVE LONG-TERM ---
    # In production, an LLM call inspects the input to extract facts.
    # Here, we programmatically catch explicit profile choices for demonstration.
    if "bullet points" in user_input.lower():
        # Write to Long-Term Memory instantly
        store.put(namespace, "user_profile", {"format": "bullet_points", "tone": "professional"})
        print(" [System Log] Captured a long-term preference pattern. Written to Store!")
        response = "Understood. I have updated your profile preferences to bullet points and will use that moving forward."
    else:
        if profile_item and profile_item.value and profile_item.value.get("format") == "bullet_points":
            response = f"• Processing request: '{user_input}'\n• Verified system updates\n• Done."
        else:
            response = f"Processed your request: '{user_input}' successfully."
            
    return {"agent_response": response}

# 3. Compile Graph with Both Memory Types
builder = StateGraph(ChatState)
builder.add_node("assistant", smart_assistant_node)
builder.add_edge(START, "assistant")
builder.add_edge("assistant", END)

# Short-term thread memory card
short_term_memory = MemorySaver()

# Long-term global database card
long_term_store = InMemoryStore()

# Compile passing BOTH components
app = builder.compile(checkpointer=short_term_memory, store=long_term_store)
print(" Adaptive Memory Graph Compiled.")

 Adaptive Memory Graph Compiled.


#Session 1 (Teaching the Agent Long-Term Context)

In [3]:
# Configuration for User 555 on a specific thread (Thread Alpha)
config_session_1 = {"configurable": {"thread_id": "thread_alpha", "user_id": "user_555"}}

input_data_1 = {
    "user_id": "user_555",
    "user_input": "Please summarize all future technical responses in bullet points.",
    "agent_response": ""
}

print(" Launching Session 1...")
result_1 = app.invoke(input_data_1, config_session_1)
print(f"\nAgent: {result_1['agent_response']}")

 Launching Session 1...
 [System Log] Active System Context: You are a helpful assistant.
 [System Log] Captured a long-term preference pattern. Written to Store!

Agent: Understood. I have updated your profile preferences to bullet points and will use that moving forward.


#Session 2 (A New Thread, Months Later — The Retrieval)

In [4]:
# Simulation: The user starts an entirely fresh chat window (Thread Omega)
# The short-term thread ID is different, but the user_id remains the same
config_session_2 = {"configurable": {"thread_id": "thread_omega", "user_id": "user_555"}}

input_data_2 = {
    "user_id": "user_555",
    "user_input": "Analyze the server logs layout query",
    "agent_response": ""
}

print(" Launching Session 2 (New Thread Windows)...")
# The agent will look at the global store, see 'user_555', and adapt its output structure automatically
result_2 = app.invoke(input_data_2, config_session_2)
print(f"\nAgent:\n{result_2['agent_response']}")

 Launching Session 2 (New Thread Windows)...
 [System Log] Active System Context: You are a helpful assistant. User Profile Context: Prefer format: bullet_points, Tone: professional.

Agent:
• Processing request: 'Analyze the server logs layout query'
• Verified system updates
• Done.
